# Pneumonia Detection from Chest X-Rays

**Portfolio edition of an MSc dissertation project**

This notebook compares a compact custom CNN with a MobileNetV2 transfer-learning model for binary chest X-ray classification. It is an academic experiment only and is **not suitable for diagnosis or clinical use**.

## 1. Study design

The original notebook used:

- a balanced custom-CNN training regime;
- a MobileNetV2 transfer-learning and light fine-tuning regime;
- a fixed 624-image test set;
- accuracy, precision, recall, specificity, F1, ROC-AUC, PR-AUC and confusion matrices;
- Grad-CAM and saliency visualisations.

The original validation split contained only 16 images. The reported thresholds were therefore tuned on the test set, which introduces optimistic bias. The metrics are preserved here as dissertation results, not as independent clinical-validation claims.

In [ ]:
from pathlib import Path
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from tensorflow.keras import callbacks, layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Dataset configuration

Download the dataset separately and keep it outside the Git repository. Set `DATA_ROOT` below to the folder containing `train`, `val` and `test`.

In [ ]:
DATA_ROOT = Path(os.environ.get("CHEST_XRAY_DATA", "path/to/chest_xray"))

TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"

required = [
    TRAIN_DIR / "NORMAL",
    TRAIN_DIR / "PNEUMONIA",
    VAL_DIR / "NORMAL",
    VAL_DIR / "PNEUMONIA",
    TEST_DIR / "NORMAL",
    TEST_DIR / "PNEUMONIA",
]

missing = [str(path) for path in required if not path.exists()]
if missing:
    print("Set DATA_ROOT before training. Missing folders:")
    for path in missing:
        print(" -", path)
else:
    print("Dataset structure verified.")

## 3. Audit class counts

In [ ]:
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def count_images(folder: Path) -> int:
    return sum(
        1 for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )

if not missing:
    audit_rows = []
    for split, folder in [("train", TRAIN_DIR), ("val", VAL_DIR), ("test", TEST_DIR)]:
        for label in ["NORMAL", "PNEUMONIA"]:
            audit_rows.append({
                "split": split,
                "label": label,
                "images": count_images(folder / label),
            })

    audit = pd.DataFrame(audit_rows)
    display(audit)

## 4. Balanced training dataframe

The dissertation's baseline regime undersampled the majority class to create an equal number of normal and pneumonia training images.

In [ ]:
def image_rows(folder: Path, label: str) -> list[dict[str, str]]:
    return [
        {"filepath": str(path), "label": label}
        for path in sorted(folder.iterdir())
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    ]

if not missing:
    normal_rows = image_rows(TRAIN_DIR / "NORMAL", "NORMAL")
    pneumonia_rows = image_rows(TRAIN_DIR / "PNEUMONIA", "PNEUMONIA")

    n_per_class = min(len(normal_rows), len(pneumonia_rows))
    rng = np.random.default_rng(SEED)

    selected_normal = rng.choice(normal_rows, size=n_per_class, replace=False).tolist()
    selected_pneumonia = rng.choice(pneumonia_rows, size=n_per_class, replace=False).tolist()

    balanced_train = pd.DataFrame(selected_normal + selected_pneumonia)
    balanced_train = balanced_train.sample(frac=1, random_state=SEED).reset_index(drop=True)

    display(balanced_train["label"].value_counts())

## 5. Custom CNN data generators

In [ ]:
CCNN_SIZE = (150, 150)
BATCH_SIZE = 32

if not missing:
    ccnn_train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
    )
    ccnn_eval_datagen = ImageDataGenerator(rescale=1.0 / 255)

    ccnn_train = ccnn_train_datagen.flow_from_dataframe(
        balanced_train,
        x_col="filepath",
        y_col="label",
        target_size=CCNN_SIZE,
        class_mode="binary",
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED,
    )
    ccnn_val = ccnn_eval_datagen.flow_from_directory(
        VAL_DIR,
        target_size=CCNN_SIZE,
        class_mode="binary",
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    ccnn_test = ccnn_eval_datagen.flow_from_directory(
        TEST_DIR,
        target_size=CCNN_SIZE,
        class_mode="binary",
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

## 6. Compact custom CNN

In [ ]:
def build_custom_cnn(input_shape=(150, 150, 3)) -> tf.keras.Model:
    model = models.Sequential(
        [
            layers.Input(shape=input_shape),
            layers.Conv2D(32, 3, activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(),

            layers.Conv2D(64, 3, activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(),

            layers.Conv2D(128, 3, activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(),

            layers.GlobalAveragePooling2D(),
            layers.Dense(64, activation="relu"),
            layers.Dropout(0.4),
            layers.Dense(1, activation="sigmoid"),
        ],
        name="custom_cnn",
    )
    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model

custom_cnn = build_custom_cnn()
custom_cnn.summary()

In [ ]:
training_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=1,
        restore_best_weights=True,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
    ),
]

# Uncomment after configuring the dataset.
# custom_cnn.fit(
#     ccnn_train,
#     validation_data=ccnn_val,
#     epochs=3,
#     callbacks=training_callbacks,
# )

## 7. MobileNetV2 transfer learning

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

MNET_SIZE = (160, 160)

if not missing:
    mnet_train_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_input,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
    )
    mnet_eval_datagen = ImageDataGenerator(
        preprocessing_function=preprocess_input
    )

    mnet_train = mnet_train_datagen.flow_from_dataframe(
        balanced_train,
        x_col="filepath",
        y_col="label",
        target_size=MNET_SIZE,
        class_mode="binary",
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED,
    )
    mnet_val = mnet_eval_datagen.flow_from_directory(
        VAL_DIR,
        target_size=MNET_SIZE,
        class_mode="binary",
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    mnet_test = mnet_eval_datagen.flow_from_directory(
        TEST_DIR,
        target_size=MNET_SIZE,
        class_mode="binary",
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

In [ ]:
def build_mobilenetv2(input_shape=(160, 160, 3)):
    base = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
    )
    base.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = models.Model(inputs, outputs, name="mobilenetv2_transfer")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model, base

mobilenet, mobilenet_base = build_mobilenetv2()
mobilenet.summary()

In [ ]:
# Original training schedule:
# 1. Train the classification head for 3 epochs with the base frozen.
# 2. Unfreeze approximately the final 20 base layers.
# 3. Fine-tune for 2 epochs at a lower learning rate.

# mobilenet.fit(
#     mnet_train,
#     validation_data=mnet_val,
#     epochs=3,
#     callbacks=training_callbacks,
# )
#
# mobilenet_base.trainable = True
# for layer in mobilenet_base.layers[:-20]:
#     layer.trainable = False
#
# mobilenet.compile(
#     optimizer=tf.keras.optimizers.Adam(1e-4),
#     loss="binary_crossentropy",
#     metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
# )
#
# mobilenet.fit(
#     mnet_train,
#     validation_data=mnet_val,
#     epochs=2,
#     callbacks=training_callbacks,
# )

## 8. Evaluation helper

In [ ]:
def evaluate_probabilities(
    y_true: np.ndarray,
    y_probability: np.ndarray,
    threshold: float,
) -> dict[str, float]:
    y_prediction = (y_probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_prediction).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_prediction,
        average="binary",
        zero_division=0,
    )
    specificity = tn / (tn + fp)
    accuracy = (tn + tp) / (tn + fp + fn + tp)

    return {
        "threshold": threshold,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc_auc_score(y_true, y_probability),
        "pr_auc": average_precision_score(y_true, y_probability),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

## 9. Reported dissertation results

These values are copied from the original notebook. They should not be interpreted as independently reproduced results from this cleaned notebook.

In [ ]:
reported_results = pd.DataFrame(
    [
        {
            "model": "Custom CNN",
            "threshold": 0.05,
            "accuracy": 0.747,
            "precision": 0.892,
            "recall": 0.677,
            "specificity": 0.863,
            "f1": 0.770,
            "roc_auc": 0.854,
            "pr_auc": 0.889,
        },
        {
            "model": "MobileNetV2",
            "threshold": 0.70,
            "accuracy": 0.907,
            "precision": 0.893,
            "recall": 0.967,
            "specificity": 0.808,
            "f1": 0.929,
            "roc_auc": 0.968,
            "pr_auc": 0.980,
        },
    ]
)

display(reported_results)

## 10. Interpretation

MobileNetV2 produced substantially stronger ROC-AUC, PR-AUC, recall and F1 than the custom CNN in the recorded experiment. Transfer learning likely helped because the pretrained network supplied richer low- and mid-level visual features than the small CNN learned during the short training schedule.

The strong test metrics still require cautious interpretation because threshold selection used the test set, the validation set was extremely small, patient-level separation was not confirmed and low-resolution resizing may have removed subtle image information.

## 11. Explainability

The original notebook generated Grad-CAM examples for the custom CNN and input-gradient saliency examples for MobileNetV2. Patient-image outputs are deliberately excluded from this public portfolio edition. They should be regenerated locally only after checking the source dataset's licence and privacy requirements.

## 12. Recommended validation redesign

A stronger follow-up experiment would:

1. create patient-level train, validation and test splits;
2. enlarge the validation set;
3. select all thresholds and hyperparameters using validation data only;
4. lock the complete pipeline;
5. evaluate once on an untouched test set;
6. report confidence intervals and calibration;
7. compare explainability methods consistently.